In [ ]:
import csv
import numpy as np
import pandas as pd
import sys, os
import re
import ast
import datetime as dt
from tqdm import tqdm
from collections import defaultdict, Counter

import warnings

warnings.filterwarnings("ignore")

from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
df = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/icustays.csv.gz')
df = df[df.los>=1]
df.shape

(74829, 8)

In [24]:
df.head(1)

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252


In [3]:
len(df['hadm_id'].unique())

68546

In [4]:
d_itemid = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/d_icd_diagnoses.csv.gz')
diagnoses_raw = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/diagnoses_icd.csv.gz')
diagnoses_raw.shape

(6364488, 5)

In [5]:
diagnoses=diagnoses_raw[diagnoses_raw['hadm_id'].isin(df['hadm_id'].unique())]

In [6]:
len(diagnoses['hadm_id'].unique())

68527

In [7]:
len(diagnoses.icd_code.unique())

17492

In [8]:
diagnoses.shape

(1281999, 5)

In [9]:
dia_iii = ['5845', '5848', '5849', '430', '431', '4321', '4329', '43301',
       '43311', '43321', '43331', '43401', '43411', '43490', '43491',
       '436', '4270', '4271', '42731', '42732', '42761', '42769', '42781',
       '42789', '7850', '5852', '5853', '5854', '5855', '5856', '5859',
       'V420', 'V4511', '490', '49120', '49121', '49122', '4919', '4920',
       '4928', '4940', '4941', '496', '27788', '2853', '3490', '34931',
       '41511', '45821', '45829', '5121', '51902', '51909', '53641',
       '53642', '53649', '56962', '5793', '78062', '99701', '99702',
       '99709', '9971', '9972', '99731', '99739', '99749', '9975',
       '99762', '99769', '99779', '99791', '99811', '99812', '99813',
       '9982', '99831', '99832', '99851', '99859', '9986', '99881',
       '99883', '99889', '9992', '99939', '99989', '9999', '41001',
       '41011', '41012', '41021', '41031', '41041', '41042', '41051',
       '41061', '41071', '41072', '41081', '41091', '41092', '4260',
       '42610', '42611', '42612', '42613', '4263', '4264', '42652',
       '42653', '4267', '42682', '42689', 'V4501', 'V4502', 'V5331',
       'V5332', '39891', '4280', '4281', '42820', '42821', '42822',
       '42823', '42830', '42831', '42832', '42833', '42840', '42841',
       '42842', '42843', '4110', '4111', '41189', '412', '4139', '41400',
       '41401', '4142', '4148', 'V4581', 'V4582', '25002', '25003',
       '25010', '25011', '25012', '25013', '25020', '25022', '25040',
       '25041', '25042', '25043', '25050', '25051', '25052', '25053',
       '25060', '25061', '25062', '25063', '25070', '25071', '25072',
       '25080', '25081', '25082', '25083', '25092', '24900', '25000',
       '25001', '79029', 'V4585', '2720', '2721', '2724', '4011', '4019',
       '2760', '2761', '2762', '2763', '2764', '27650', '27651', '27652',
       '27669', '2767', '2768', '2769', '9951', '4560', '45620', '5307',
       '53082', '53100', '53140', '53200', '53240', '53440', '5693',
       '5780', '5781', '5789', '4010', '40291', '40300', '40301', '40310',
       '40311', '40390', '40391', '40491', '40493', '4372', '570', '5715',
       '5716', '5718', '5720', '5722', '5723', '5724', '5728', '5730',
       '5734', '5738', '5739', '7824', '78959', '7904', '7905', '7948',
       'V427', '514', '515', '5168', '5178', '5183', '5184', '51889',
       '5194', '5198', '78603', '78605', '78606', '78609', '7862',
       '78630', '78639', '78652', '7866', '7868', '79311', '4779',
       '47824', '47829', '47830', '47831', '4785', '4786', '47874',
       '47875', '51919', '5192', '78442', '7847', '7861', 'V440', 'V550',
       '5100', '5109', '5110', '5111', '51189', '5119', '5120', '5180',
       '5181', '1124', '1363', '4809', '481', '4820', '4821', '4822',
       '48240', '48241', '48242', '48282', '48283', '48284', '4829',
       '4846', '485', '486', '5130', '51851', '51852', '51881', '51882',
       '51883', '51884', '7991', 'V4611', 'V462', '0380', '03810',
       '03811', '03812', '03819', '0382', '0383', '03840', '03842',
       '03843', '03844', '03849', '0388', '0389', '449', '77181', '7907',
       '99591', '99592', '78550', '78551', '78552', '78559', '00845',
       '07054', '2449', '2639', '2749', '27800', '27801', '2800', '2809',
       '2851', '28521', '28529', '2859', '2869', '2875', '2930', '2948',
       '30000', '3004', '3051', '311', '32723', '3572', '4168', '4240',
       '4241', '4254', '4275', '4439', '4589', '49390', '5070', '53081',
       '5601', '5712', '5990', '60000', '70703', '73300', '78039',
       '78791', '78820', '79902', 'E8497', 'E8782', 'E8788', 'E8798',
       'V053', 'V103', 'V1046', 'V1251', 'V1254', 'V1582', 'V290',
       'V4986', 'V502', 'V5861', 'V5867', 'V667', '30500', 'E8889',
       '34590', '79092', '3485', '71590', '5770', 'V5865', '99662',
       '28860', '36201', '56210']

diags = ['A419', 'A4189', 'A4151', 'A4102', 'A4181', 'A4152', 'A4150','A4159', 'A411', 'A4101', 'A414', 'A4153', 'A413', 'A412','I95', 'E87', 'R10', 'F10', 'R68', 'J44', 'K70', 'J69', 'I50', 'J98', 'N17', 'K92', 'R09', 'I47', 'D64', 'I25', 'M15', 'M81', 'F41', 'I10', 'I34', 'I21', 'N18', 'I12', 'E11', 'Z51', 'K22', 'G93', 'J45', 'F32', 'Z91', 'I20', 'G61', 'E78', 'E66', 'Z95', 'J18', 'J41', 'R69', 'R00', 'R19', 'D50', 'A04', 'E89', 'A40', 'N39', 'T78', 'K56', 'J47', 'K57', 'R78', 'K76', 'D69', 'I42', 'I27', 'B15', 'J17', 'T81', 'G47', 'E44', 'F05', 'M10', 'L89', 'K26', 'I44', 'K31', 'J15', 'I24', 'Z85', 'I66', 'G44', 'K75', 'I85', 'I61', 'K85', 'D66', 'N23', 'K25', 'D70', 'J93', 'H35', 'J30', 'N40', 'G40', 'R06', 'I73', 'F04', 'J81', 'K62', 'R20', 'I26', 'R45', 'T88', 'J84', 'I60', 'B25', 'J43', 'J13', 'T82', 'E88', 'I62', 'R94', 'G97', 'J86', 'J34', 'J95', 'E84', 'L94', 'I76', 'I67', 'I11', 'J40', 'Z23', 'K90', 'J85', 'B37', 'K28', 'J12', 'J96', 'Y92', 'A41']
add_diag = ['B343', 'B677', 'Z8614', 'B341', 'A9230', 'B669', '9583', '0066', 'B674', '01172', '01175', 'J9383', 'S270XXS', 'S270XXA', 'A493', '5121', 'B344', 'B342', '01174', '01176', 'A749', '1049', 'L0201', '05910', 'A549', '1224', '51281', 'A4902', 'A492', 'A699', '51283', '8600', 'S270XXD', 'K8590', '5120', 'A491', 'S272', 'J9581', 'J930', 'A490', 'S272XXS', 'J938', 'J069', 'B349', '01170', '01173', 'A02', 'S270', 'A084', 'B009', 'B340', 'J9312', 'K922', 'J95811', '5128', 'A049', 'A399', 'S272XXA', 'A609', 'J931', 'S272XXD', 'J9311', 'A319', 'J9381', '01171', '51289', 'B0860', 'A499', '8601', 'A4901', '51282', 'B0870', '0369', '1227', '05920']

all_diag = diags+dia_iii+add_diag
all_diag = pd.unique(all_diag)
print(len(all_diag),len(diags),len(dia_iii),len(add_diag))

612 135 404 75


In [ ]:
diag_9_10 = pd.read_csv('/0_diag_9_10.csv')

In [15]:
diagnoses.head(2)

,subject_id,hadm_id,seq_num,icd_code,icd_version
115,10000690,25860671,1,5070,9
116,10000690,25860671,2,42833,9


In [16]:
diagnoses=diagnoses[diagnoses['icd_code'].isin(all_diag)]

In [17]:
len(diagnoses.icd_code.unique())

457

In [19]:
diagnoses[diagnoses.icd_code.str.contains('A41')].head(2)

,subject_id,hadm_id,seq_num,icd_code,icd_version
2830,10003400,23559586,5,A419,10
3175,10003637,28317408,1,A4189,10


In [20]:
diagnoses_9 = pd.merge(diagnoses[diagnoses.icd_version==9],diag_9_10,on='icd_code',how='left')
diagnoses_9 = diagnoses_9.drop_duplicates(subset=['hadm_id','icd_code', 'root'], keep='first')
diagnoses_9['root'] = diagnoses_9['root'].fillna(diagnoses_9['icd_code'])

In [21]:
diagnoses_10 = pd.merge(diagnoses[diagnoses.icd_version==10],diag_9_10,on='icd_code',how='left')
diagnoses_10 = diagnoses_10.drop_duplicates(subset=['hadm_id','icd_code', 'root'], keep='first')
diagnoses_10['root'] = diagnoses_10['root'].fillna(diagnoses_10['icd_code'])

In [22]:
print(diagnoses_9.shape,diagnoses_10.shape)

(338029, 6) (27020, 6)


In [23]:
diagnoses_9[diagnoses_9.duplicated(subset=['hadm_id','icd_code', 'root'], keep=False)]

,subject_id,hadm_id,seq_num,icd_code,icd_version,root


In [25]:
diagnoses_grouped_9 = diagnoses_9.groupby(['hadm_id', 'root'])['icd_code'].apply(list).reset_index()
diagnoses_grouped_10 = diagnoses_10.groupby(['hadm_id', 'root'])['icd_code'].apply(list).reset_index()

In [26]:
result_9 = diagnoses_grouped_9.pivot(index='hadm_id', columns='root', values='icd_code')
result_10 = diagnoses_grouped_10.pivot(index='hadm_id', columns='root', values='icd_code')

In [27]:
result_9 = result_9.reset_index()
result_10 = result_10.reset_index()

In [29]:
all_result = pd.concat([result_9,result_10])

In [32]:
missing_9 = result_9.isna().mean() * 100
len(missing_9[missing_9<95])

50

In [33]:
missing_10 = result_10.isna().mean() * 100
len(missing_10[missing_10<95])

5

In [34]:
missing_ratio = all_result.isna().mean() * 100

In [36]:
importv = ['J15', 'L89', 'A40', 'N39', 'T78', 'J95','T81','A41','K62', 'J18','J44','J15',
'L89', 'T78', 'A40','N39','L94', 'K28','J40', 'J15', 'J30','J13','I20','J47',
'N39','I76', 'Z23', 'F04']
print(len(importv))
importv = pd.unique(importv)
len(importv)

28


21

In [232]:
final_diags = missing_ratio[missing_ratio<95].index.tolist()
final_diags = pd.unique(final_diags)
print(final_diags)
print(len(final_diags))

['hadm_id' 'A40' 'D64' 'D69' 'E11' 'E66' 'E78' 'E8497' 'E87' 'E8798' 'E89'
 'F05' 'F10' 'F32' 'F41' 'G47' 'I10' 'I12' 'I21' 'I25' 'I27' 'I34' 'I47'
 'I50' 'I95' 'J18' 'J44' 'J69' 'J98' 'K22' 'N17' 'N18' 'N39' 'R00' 'R09'
 'R68' 'R78' 'T78' 'T81' 'Z51' 'Z91' 'Z95' 'A41']
43


In [38]:
all_result = all_result[['hadm_id','A40','D64', 'D69' ,'E11', 'E66' ,'E78', 'E8497', 'E87', 'E8798', 'E89',
 'F05', 'F10', 'F32', 'F41', 'G47', 'I10', 'I12', 'I21', 'I25', 'I27', 'I34', 'I47',
 'I50', 'I95', 'J18', 'J44', 'J69', 'J98', 'K22', 'N17', 'N18', 'N39', 'R00', 'R09',
 'R68', 'R78', 'T78', 'T81', 'Z51', 'Z91', 'Z95', 'A41']]

In [39]:
all_result.shape

(53533, 43)

In [40]:
all_result.head()

root,hadm_id,A40,D64,D69,E11,E66,E78,E8497,E87,E8798,...,R00,R09,R68,R78,T78,T81,Z51,Z91,Z95,A41
0,20000808,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20001361,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[2760],NaN,...,[78550],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20002356,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20004004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[2762, 27652]",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[V4582],NaN
4,20004357,NaN,[2859],NaN,NaN,NaN,[2724],NaN,[2764],NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,[V5865],NaN,NaN,NaN


In [ ]:
all_result.to_csv('/MIMIC_IV/All_diags_nan.csv',index=False)

In [42]:
binary_all_result = all_result.copy()
binary_all_result.loc[:, binary_all_result.columns != 'hadm_id'] = binary_all_result.loc[:, binary_all_result.columns != 'hadm_id'].notna().astype(int)

In [43]:
binary_all_result

root,hadm_id,A40,D64,D69,E11,E66,E78,E8497,E87,E8798,...,R00,R09,R68,R78,T78,T81,Z51,Z91,Z95,A41
0,20000808,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,20001361,0,0,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0
2,20002356,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,20004004,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
4,20004357,0,1,0,0,0,1,0,1,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21137,29998113,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21138,29998399,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21139,29998464,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
21140,29998928,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
binary_all_result.to_csv('/MIMIC_IV/All_diags_binary.csv',index=False)

In [46]:
final_diags = ['A40','D64', 'D69' ,'E11', 'E66' ,'E78', 'E8497', 'E87', 'E8798', 'E89',
 'F05', 'F10', 'F32', 'F41', 'G47', 'I10', 'I12', 'I21', 'I25', 'I27', 'I34', 'I47',
 'I50', 'I95', 'J18', 'J44', 'J69', 'J98', 'K22', 'N17', 'N18', 'N39', 'R00', 'R09',
 'R68', 'R78', 'T78', 'T81', 'Z51', 'Z91', 'Z95', 'A41']